# Pose estimation

## Sources:
- [Website title](www.some_website.com)

## Notes:
- reducing jittering with low pass filter make pose less reactive to movements (the pose doesn't follow the body when a movement is quick) => reduce window size to improve reactivity
- there are still outliers => try interpolation

## Import modules

In [13]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
import time

# import 3rd-party modules
import numpy as np
import cv2
import mediapipe as mp
from imutils.video import FileVideoStream
from imutils.video import FPS
from matplotlib import pyplot as plt

# import local modules


## Define CONSTANTS & variables

In [14]:
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic
# mp_face_mesh = mp.solutions.face_mesh

In [19]:
KEYPOINT_DICT = {
    "nose": 0,
    "left_eye_inner": 1,
    "left_eye": 2,
    "left_eye_outer": 3,
    "right_eye_inner": 4,
    "right_eye": 5,
    "right_eye_outer": 6,
    "left_ear": 7,
    "right_ear": 8,
    "mouth_left": 9,
    "mouth_right": 10,
    "left_shoulder": 11,
    "right_shoulder": 12,
    "left_elbow": 13,
    "right_elbow": 14,
    "left_wrist": 15,
    "right_wrist": 16,
    "left_pinky": 17,
    "right_pinky": 18,
    "left_index": 19,
    "right_index": 20,
    "left_thumb": 21,
    "right_thumb": 22,
    "left_hip": 23,
    "right_hip": 24,
    "left_knee": 25,
    "right_knee": 26,
    "left_ankle": 27,
    "right_ankle": 28,
    "left_heel": 29,
    "right_heel": 30,
    "left_foot_index": 31,
    "right_foot_index": 32
}

AXIS_DICT = {
    "x": 0,
    "y": 1,
    "z": 2
}


In [ ]:
# get list of connections between keypoints (to draw the lines)
pose_connections_list = list(mp_holistic.POSE_CONNECTIONS)
# pose_connections = []
# connections_len = len(mp_holistic.POSE_CONNECTIONS)
# for i in range(connections_len):
#     pose_connections.append((pose_connections_list[i][0].value, pose_connections_list[i][1].value))

## Define functions

In [ ]:
def draw_keypoints(keypoints_xs, keypoints_ys, keypoints_connections,
                        image_height, image_width, image=None,
                        dot_color_bgr=(255,0,0), line_color_bgr=(0,0,255),
                        circle_radius = 5, line_thickness = 1
                    ) -> np.array:
    """
    Function to draw pose keypoints and connections between them.
    
    Retuns: image with keypoints and connections drawn
    """
    
    # loop through all keypoints
    for idx in range(len(keypoints_xs)):
        
        # get xy coords
        xy = (keypoints_xs[idx], keypoints_ys[idx])
        
        # draw circles
        cv2.circle(image,
                tuple(np.multiply(xy, (image_width, image_height)).astype(int)), # center coordinates
                circle_radius, # radius of circle
                dot_color_bgr, # color in bgr
                line_thickness # line thickness
                )

    # loop through all keypoints connections
    for start_kp_idx, end_kp_idx in keypoints_connections:
        
        # get xy coords of line start and end
        start_line = (keypoints_xs[start_kp_idx], keypoints_ys[start_kp_idx])
        end_line = (keypoints_xs[end_kp_idx], keypoints_ys[end_kp_idx])
        
        # draw line
        cv2.line(image,
                tuple(np.multiply(start_line, (image_width, image_height)).astype(int)),
                tuple(np.multiply(end_line, (image_width, image_height)).astype(int)),
                line_color_bgr, # color in bgr
                line_thickness # line thickness
                )
    
    return image

## Run logic

In [3]:
# set video path
video_path = Path("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images/gagu/classic.mp4")
# video_path = 1 # webcam number

# set empty list to store xyz landmarks
landmarks_xyzs = []

In [20]:
video_cap = cv2.VideoCapture(str(video_path))

with mp_holistic.Holistic(
  min_detection_confidence=0.7,
  min_tracking_confidence=0.7) as holistic:

  while video_cap.isOpened():
    success, image = video_cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      if isinstance(video_path, int):
        continue
      else:
        break

    # To improve performance, optionally mark the image as not writeable to
    # pass by reference.
    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = holistic.process(image)

    # Draw landmark annotation on the image.
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    mp_drawing.draw_landmarks(
        image,
        results.face_landmarks,
        mp_holistic.FACEMESH_CONTOURS,
        landmark_drawing_spec=None,
        connection_drawing_spec=mp_drawing_styles
        .get_default_face_mesh_contours_style())
    mp_drawing.draw_landmarks(
        image,
        results.pose_landmarks,
        mp_holistic.POSE_CONNECTIONS,
        landmark_drawing_spec=mp_drawing_styles
        .get_default_pose_landmarks_style())

    # Flip the image horizontally for a selfie-view display.
    cv2.imshow('MediaPipe Holistic', cv2.flip(image, 1))
    if cv2.waitKey(5) & 0xFF == 27:
      break

    landmarks_xyzs.append([(landmark.x, landmark.y, landmark.z) for landmark in results.pose_landmarks.landmark])
    # results.multi_face_landmarks[0].landmark]

video_cap.release()

cv2.destroyAllWindows()
cv2.waitKey(1)

Ignoring empty camera frame.


-1

## Save landmarks

In [11]:
# convert list to numpy array
landmarks_xyzs_array = np.array(landmarks_xyzs)

# set landmarks array path
landmarks_xyzs_array_path = video_path.parent / f"{video_path.stem}.npy"

# save landmarks array
np.save(landmarks_xyzs_array_path, landmarks_xyzs_array)

## Load landmarks

In [ ]:
# set landmarks array path
landmarks_xyzs_array_path = video_path.parent / f"{video_path.stem}.npy"
print(landmarks_xyzs_array_path)

# save landmarks array
landmarks_xyzs_array = np.load(landmarks_xyzs_array_path)
print(landmarks_xyzs_array.shape)

## Define Functions to reduce jittering of pose estimation

### Exponential smoothing

#### Chatgpt:
To reduce jittering in mediapipe pose landmarks, you can use a smoothing technique such as exponential smoothing. Exponential smoothing works by applying a smoothing factor to the current value and the previous smoothed value, so that the output value is a weighted average of the current value and the previous smoothed value.

Here is an example of how you can implement exponential smoothing in Python:



In [ ]:
def exponential_smoothing(alpha, data):
    smoothed_data = []
    for i in range(len(data)):
        if i == 0:
            smoothed_data.append(data[i])
        else:
            smoothed_data.append(alpha * data[i] + (1 - alpha) * smoothed_data[i-1])
    return smoothed_data

# Example usage
alpha = 0.5  # Smoothing factor
data = [1, 2, 3, 4, 5, 6]
smoothed_data = exponential_smoothing(alpha, data)
print(smoothed_data)  # Output: [1, 1.5, 2.25, 3.125, 4.0625, 5.03125]


In this example, the alpha parameter determines the amount of smoothing applied. A larger value of alpha will result in more smoothing, while a smaller value will result in less smoothing.

You can adjust the value of alpha to find the best balance between smoothing and preserving the original signal. You can also experiment with different smoothing techniques, such as moving average or low pass filtering, to see which one works best for your specific application.

### Low pass filter
To implement a low pass filter in Python, you can use a simple moving average. A moving average works by taking the average of a set of consecutive data points and using that average as the output value. This has the effect of smoothing the data and removing high frequency noise.

Here is an example of how you can implement a moving average low pass filter in Python:

In [5]:
def low_pass_filter(data, window_size):
    filtered_data = []
    for i in range(len(data)):
        if i < window_size:
            filtered_data.append(sum(data[:i+1]) / (i+1))
        else:
            filtered_data.append(sum(data[i-window_size+1:i+1]) / window_size)
    return filtered_data

# Example usage
data = [1, 2, 3, 4, 5, 6]
window_size = 3  # Size of the moving average window
filtered_data = low_pass_filter(data, window_size)
print(filtered_data)  # Output: [1, 1.5, 2.0, 3.0, 4.0, 5.0]

[1.0, 1.5, 2.0, 3.0, 4.0, 5.0]


In this example, the window_size parameter determines the number of data points to include in the moving average. A larger window size will result in more smoothing, while a smaller window size will preserve more of the original signal.

You can adjust the value of window_size to find the best balance between smoothing and preserving the original signal. You can also experiment with different window sizes to see which one works best for your specific application.

In [9]:
# Example usage
data = [1, 2, 3, 4, 5, 6]
window_size = 3  # Size of the moving average window

filtered_data = []
for i in range(len(data)):
    print(i)
    if i < window_size:
        print(data[:i+1], (i+1))
        filtered_data.append(sum(data[:i+1]) / (i+1))
    else:
        print(data[i-window_size+1:i+1], window_size)
        filtered_data.append(sum(data[i-window_size+1:i+1]) / window_size)

    print(filtered_data)


0
[1] 1
[1.0]
1
[1, 2] 2
[1.0, 1.5]
2
[1, 2, 3] 3
[1.0, 1.5, 2.0]
3
[2, 3, 4] 3
[1.0, 1.5, 2.0, 3.0]
4
[3, 4, 5] 3
[1.0, 1.5, 2.0, 3.0, 4.0]
5
[4, 5, 6] 3
[1.0, 1.5, 2.0, 3.0, 4.0, 5.0]


### Remove outliers
To filter out outliers in mediapipe pose landmarks, you can use a technique called statistical outlier removal. This involves identifying points that are significantly different from the rest of the data and removing them.

Here is an example of how you can implement statistical outlier removal in Python:

In [ ]:
def remove_outliers(data, m=2):
    data = np.array(data)
    mean = np.mean(data)
    std = np.std(data)
    filtered_data = data[abs(data - mean) <= m * std]
    return filtered_data

# Example usage
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
filtered_data = remove_outliers(data)
print(filtered_data)  # Output: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In this example, the m parameter determines the number of standard deviations away from the mean that a point must be to be considered an outlier. A larger value of m will result in more points being removed, while a smaller value will result in fewer points being removed.

You can adjust the value of m to find the best balance between removing outliers and preserving the original data. You can also experiment with different outlier removal techniques, such as the interquartile range method, to see which one works best for your specific application.

### Interpolate outliers
To replace outliers in mediapipe pose landmarks with interpolated data, you can use a technique called linear interpolation. Linear interpolation works by fitting a straight line between two known points and using that line to estimate the value at any other point within the range of those two points.

Here is an example of how you can implement linear interpolation in Python:

In [14]:
def interpolate_outliers(data, m=2):
    data = np.array(data)
    mean = np.mean(data)
    std = np.std(data)
    outliers = data[abs(data - mean) > m * std]
    for outlier in outliers:
        index = np.where(data == outlier)[0][0]
        if index == 0:
            data[index] = (data[index+1] + data[index]) / 2
        elif index == len(data) - 1:
            data[index] = (data[index-1] + data[index]) / 2
        else:
            data[index] = (data[index-1] + data[index+1]) / 2
    return data

# Example usage
data = [1, 2, 3, 4, 5, 20, 7, 8, 9, 10]
# data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
interpolated_data = interpolate_outliers(data)
# print(interpolated_data)  # Output: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print(interpolated_data)

[ 1  2  3  4  5  6  7  8  9 10]


In this example, the m parameter determines the number of standard deviations away from the mean that a point must be to be considered an outlier. A larger value of m will result in more points being replaced, while a smaller value will result in fewer points being replaced.

You can adjust the value of m to find the best balance between replacing outliers and preserving the original data. You can also experiment with different interpolation methods, such as polynomial interpolation or spline interpolation, to see which one works best for your specific application.

### Replace outliers with linspace

In [ ]:
def find_outliers_idxs(data, m=2):
    data = np.array(data)
    mean = np.mean(data)
    std = np.std(data)
    outliers_idxs = (abs(data - mean) > m * std).nonzero()

    return outliers_idxs


def replace_outliers(data, m=0.2):

    gradient_data = np.gradient(data)
    outliers_idxs = find_outliers_idxs(gradient_data, m=m)

    # print(outliers_idxs)

    diff = np.diff(outliers_idxs[0], prepend=outliers_idxs[0][0])
    segment = np.cumsum(diff > 1)

    unique_segment = np.unique(segment)

    segment_start_end_idxs_array = []
    segment_start_end_array = []

    for segment_nb in unique_segment:
        segment_start_end_idxs = np.argwhere(segment == segment_nb)[[0,-1]]
        segment_start_end_idxs_array.append(segment_start_end_idxs)

        segment_start_end = outliers_idxs[0][segment_start_end_idxs]
        segment_start_end_array.append(segment_start_end)

    corrected_data = data.copy()

    for segment_start_end in segment_start_end_array:
        start_idx = segment_start_end[0][0]
        end_idx = segment_start_end[1][0]
        num = end_idx - start_idx

        # print(start_idx, end_idx, num)
        corrected_data[start_idx:end_idx] = np.linspace(data[start_idx], data[end_idx], num=num)

    return corrected_data

### Find peaks and troughs

In [ ]:
def detect_peaks_and_troughs(data, lookahead=500, threshold=0.05):
    """
    Detect peaks and troughs in time series data using a simple algorithm.
    :param data: Time series data (numpy array)
    :param lookahead: Number of data points to consider before and after a peak or trough
    :param threshold: Minimum relative amplitude of a peak or trough to be considered a peak or trough
    :return: Tuple of lists of indices of detected peaks and troughs
    """
    peaks = []
    troughs = []
    for i in range(len(data)):
        if i < lookahead:
            continue
        elif i > len(data) - lookahead:
            break
        elif data[i] > threshold * np.max(data):
            peaks.append(i)
        elif data[i] < threshold * np.min(data):
            troughs.append(i)
    return peaks, troughs

### Smooth by convolution

In [ ]:
def smooth(y, box_pts):
    box = np.ones(box_pts)/box_pts
    y_smooth = np.convolve(y, box, mode='same')
    return y_smooth

## Run functions to reduce jittering of pose estimation

In [21]:
# test functions on a keypoint
AXIS_STR = "y" # x,y or z
AXIS_NB = AXIS_DICT[AXIS_STR]
KEYPOINT_STR = "left_heel"
KEYPOINT_NB = KEYPOINT_DICT[KEYPOINT_STR]
test_data = landmarks_xyzs_array[:,KEYPOINT_NB,AXIS_NB]

NB_STD_AWAY_FROM_MEAN = 0.2
WINDOW_LEN = 20

outliers_idxs = find_outliers_idxs(data, m=NB_STD_AWAY_FROM_MEAN)

# plot keypoint positions across landmarks

plt.title(f"{KEYPOINT_STR}")
plt.xlabel("frame number")
plt.ylabel(f"{KEYPOINT_STR} {AXIS_STR} position")

plt.plot(test_data)
plt.scatter(outliers_idxs[0], test_data[outliers_idxs[0]])

extraticks = [0, 50, 79, 89, 150, 200, 250] # outliers_idxs[0]
# plt.xticks(list(plt.xticks()[0]) + extraticks)
plt.xticks(extraticks)
plt.grid()

plt.show()

NameError: name 'plt' is not defined

In [ ]:
# np.std(test_data)/0.1


interpolated_outliers_test_data = np.array([])
replaced_outliers_test_data = np.array([])

for i in np.arange(0,len(test_data),WINDOW_LEN,dtype=int):
    interpolated_outliers_test_data = np.append(interpolated_outliers_test_data, interpolate_outliers(test_data[i:i+WINDOW_LEN], NB_STD_AWAY_FROM_MEAN))
    replaced_outliers_test_data = np.append(replaced_outliers_test_data, replace_outliers(test_data[i:i+WINDOW_LEN], NB_STD_AWAY_FROM_MEAN))

plt.plot(test_data)
plt.plot(interpolated_outliers_test_data)
plt.plot(replaced_outliers_test_data)

plt.xticks(extraticks)

plt.grid()

plt.show()

In [ ]:
interprated_outliers_landmarks_xyzs_array = np.empty_like(landmarks_xyzs_array)


for axis in range(landmarks_xyzs_array.shape[2]):
    for landmark_idx in range(landmarks_xyzs_array.shape[1]):
        data = landmarks_xyzs_array[:,landmark_idx,axis]
        for i in np.arange(0,len(data),WINDOW_LEN,dtype=int):
            interprated_outliers_landmarks_xyzs_array[i:i+WINDOW_LEN,landmark_idx,axis] = interpolate_outliers(data[i:i+WINDOW_LEN], m)

# set landmarks array path
interprated_outliers_landmarks_xyzs_array_path = video_path.parent / f"{video_path.stem}_interprated_outliers.npy"
print(interprated_outliers_landmarks_xyzs_array_path)

# save landmarks array
np.save(interprated_outliers_landmarks_xyzs_array_path, interprated_outliers_landmarks_xyzs_array)

In [ ]:
plt.plot(np.gradient(test_data))

# plt.xticks(list(plt.xticks()[0]) + extraticks)
plt.xticks(extraticks)

plt.grid()

plt.show()

In [ ]:
smooth_landmarks_xyzs_array = np.empty_like(landmarks_xyzs_array)

for axis in range(landmarks_xyzs_array.shape[2]):
    for landmark_idx in range(landmarks_xyzs_array.shape[1]):
        smooth_landmarks_xyzs_array[:,landmark_idx,axis] = low_pass_filter(landmarks_xyzs_array[:,landmark_idx,axis], 3)

In [23]:
smooth_landmarks_xyzs_array = np.empty_like(landmarks_xyzs_array)
window_size = 3

for axis in range(landmarks_xyzs_array.shape[2]):
    for landmark_idx in range(landmarks_xyzs_array.shape[1]):
        smooth_landmarks_xyzs_array[:,landmark_idx,axis] = low_pass_filter(landmarks_xyzs_array[:,landmark_idx,axis], window_size)

In [27]:
convolved_landmarks_xyzs_array = np.empty_like(landmarks_xyzs_array)
box_points = 3

for axis in range(landmarks_xyzs_array.shape[2]):
    for landmark_idx in range(landmarks_xyzs_array.shape[1]):
        convolved_landmarks_xyzs_array[:,landmark_idx,axis] = smooth(landmarks_xyzs_array[:,landmark_idx,axis], box_points)

        # get original starting and ending pose
        # the convolved versions are really out of place
        convolved_landmarks_xyzs_array[0,landmark_idx,axis] = landmarks_xyzs_array[0,landmark_idx,axis]
        convolved_landmarks_xyzs_array[-1,landmark_idx,axis] = landmarks_xyzs_array[-1,landmark_idx,axis]

In [31]:
interprated_outliers_landmarks_xyzs_array = np.empty_like(landmarks_xyzs_array)
m = 3

for axis in range(landmarks_xyzs_array.shape[2]):
    for landmark_idx in range(landmarks_xyzs_array.shape[1]):
        interprated_outliers_landmarks_xyzs_array[:,landmark_idx,axis] = interpolate_outliers(landmarks_xyzs_array[:,landmark_idx,axis], box_points)

## Draw original and modified keypoints

In [12]:
WINDOW_NAME = "frame" # set window name
QUEUE_SIZE = 128 # set queue size (i.e. buffer)

# set output video codec
# use avc1 instead of h264 for mp4 video as tag 0x34363268/'h264' is not supported with codec and mp4 format
CODEC = "avc1"

In [32]:
# set input & output video path
video_path = Path("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images/gagu/classic.mp4")
out_path = f"{video_path.parent}/{video_path.stem}_with&without_smooth_{window_size}_&convolved_{box_points}_&interprate_outliers_{m}.mp4"

# initialize video stream
# pass transform fct to process frame in file video stream thread
fvs = FileVideoStream(path=str(video_path), transform=None, queue_size=QUEUE_SIZE)

# get video stream parameters
video_n_frames = int(fvs.stream.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = fvs.stream.get(cv2.CAP_PROP_FPS)
video_width = int(fvs.stream.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(fvs.stream.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_nb = int(fvs.stream.get(cv2.CAP_PROP_POS_FRAMES))

print(f"number of frames = {video_n_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*CODEC)
out_video = cv2.VideoWriter(filename=out_path, fourcc=fourcc, fps=video_fps, frameSize=(video_width, video_height))

# start file video stream thread and allow buffer to
# start to fill
print("[INFO] starting video file thread...")
fvs.start()
time.sleep(1.0)

# start fps timer to monitor runtime
fps = FPS().start()

# loop over frames from video file stream
while fvs.more():

	# grab frame from the threaded video file stream
	frame = fvs.read()
	if frame is None:
		print("Ignoring empty camera frame.")
		break

	# process frame
	frame = draw_keypoints(
        smooth_landmarks_xyzs_array[frame_nb,:,0],
        smooth_landmarks_xyzs_array[frame_nb,:,1],
        mp_holistic.POSE_CONNECTIONS,
        video_height, video_width, 
        frame,
        dot_color_bgr=(0,0,255),
        line_color_bgr=(255,255,255)
        # dot_color_bgr=tuple(np.random.randint(0,255,size=3).tolist()),
        # line_color_bgr=tuple(np.random.randint(0,255,size=3).tolist())
        )

	frame = draw_keypoints(
		landmarks_xyzs_array[frame_nb,:,0],
		landmarks_xyzs_array[frame_nb,:,1],
		mp_holistic.POSE_CONNECTIONS,
		video_height, video_width, 
		frame,
        dot_color_bgr=(255,0,0),
        line_color_bgr=(127,127,127)
		# dot_color_bgr=tuple(np.random.randint(0,255,size=3).tolist()),
		# line_color_bgr=tuple(np.random.randint(0,255,size=3).tolist())
		)

	frame = draw_keypoints(
		convolved_landmarks_xyzs_array[frame_nb,:,0],
		convolved_landmarks_xyzs_array[frame_nb,:,1],
		mp_holistic.POSE_CONNECTIONS,
		video_height, video_width, 
		frame,
        dot_color_bgr=(0,255,0),
        line_color_bgr=(0,100,50)
		# dot_color_bgr=tuple(np.random.randint(0,255,size=3).tolist()),
		# line_color_bgr=tuple(np.random.randint(0,255,size=3).tolist())
		)

	frame = draw_keypoints(
		interprated_outliers_landmarks_xyzs_array[frame_nb,:,0],
		interprated_outliers_landmarks_xyzs_array[frame_nb,:,1],
		mp_holistic.POSE_CONNECTIONS,
		video_height, video_width, 
		frame,
        dot_color_bgr=(0,255,255),
        line_color_bgr=(0,255,255)
		# dot_color_bgr=tuple(np.random.randint(0,255,size=3).tolist()),
		# line_color_bgr=tuple(np.random.randint(0,255,size=3).tolist())
		)

	# show frame
	cv2.imshow(WINDOW_NAME, frame)

    # wait for a key 
    # 0xFF to check what key we pressed on the keyboard
	key = cv2.waitKey(10) & 0xFF

    # break out of stream loop if esc or 'q' is pressed
	if key == 27 or key == ord('q'):        
		break
	
	# write output frame
	out_video.write(frame)

	# update fps counter
	fps.update()

	frame_nb += 1
	
# stop timer and display FPS information
fps.stop()
print("[INFO] elasped time: {:.2f}".format(fps.elapsed()))
print("[INFO] approx. FPS: {:.2f}".format(fps.fps()))

# close windows
cv2.destroyAllWindows()
cv2.waitKey(1) # workaround to effectively close window on mac

# release video stream & video rendering
fvs.stop()
out_video.release()

number of frames = 752
fps = 30.02294306464692
video width = 720
video height = 1280
[INFO] starting video file thread...
Ignoring empty camera frame.
[INFO] elasped time: 13.36
[INFO] approx. FPS: 56.28


: 

In [10]:
m = 0.2
landmarks_xyzs_array = np.load("/Users/derrickvanfrausum/BeCode_AI/git-repos/sports-vision/object_detection/data/video_inputs/federer-slow-motion-backhand-trim.npy")
interprated_outliers_landmarks_xyzs_array = np.load("/Users/derrickvanfrausum/BeCode_AI/git-repos/sports-vision/object_detection/data/video_inputs/federer-slow-motion-backhand-trim_interprated_outliers.npy")

In [17]:
# set input & output video path
video_path = Path("/Users/derrickvanfrausum/BeCode_AI/git-repos/sports-vision/object_detection/data/video_inputs/federer-slow-motion-backhand-trim_processed.mp4")
out_path = f"{video_path.parent}/{video_path.stem}_with&without_interprate_outliers_{m}.mp4"

# initialize video stream
# pass transform fct to process frame in file video stream thread
fvs = FileVideoStream(path=str(video_path), transform=None, queue_size=QUEUE_SIZE)

# get video stream parameters
video_n_frames = int(fvs.stream.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = fvs.stream.get(cv2.CAP_PROP_FPS)
video_width = int(fvs.stream.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(fvs.stream.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_nb = int(fvs.stream.get(cv2.CAP_PROP_POS_FRAMES))

print(f"number of frames = {video_n_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*CODEC)
out_video = cv2.VideoWriter(filename=out_path, fourcc=fourcc, fps=video_fps, frameSize=(video_width, video_height))

# start file video stream thread and allow buffer to
# start to fill
print("[INFO] starting video file thread...")
fvs.start()
time.sleep(1.0)

# start fps timer to monitor runtime
fps = FPS().start()

# loop over frames from video file stream
while fvs.more():

	# grab frame from the threaded video file stream
	frame = fvs.read()
	if frame is None:
		print("Ignoring empty camera frame.")
		break

	# process frame
	frame = draw_keypoints(
        landmarks_xyzs_array[frame_nb,:,0],
        landmarks_xyzs_array[frame_nb,:,1],
        mp_holistic.POSE_CONNECTIONS,
        video_height, video_width, 
        frame,
        dot_color_bgr=(0,0,255),
        line_color_bgr=(255,255,255)
        # dot_color_bgr=tuple(np.random.randint(0,255,size=3).tolist()),
        # line_color_bgr=tuple(np.random.randint(0,255,size=3).tolist())
        )

	frame = draw_keypoints(
		interprated_outliers_landmarks_xyzs_array[frame_nb,:,0],
		interprated_outliers_landmarks_xyzs_array[frame_nb,:,1],
		mp_holistic.POSE_CONNECTIONS,
		video_height, video_width, 
		frame,
        dot_color_bgr=(255,0,0),
        line_color_bgr=(127,127,127)
		# dot_color_bgr=tuple(np.random.randint(0,255,size=3).tolist()),
		# line_color_bgr=tuple(np.random.randint(0,255,size=3).tolist())
		)

	# show frame
	cv2.imshow(WINDOW_NAME, frame)

    # wait for a key 
    # 0xFF to check what key we pressed on the keyboard
	key = cv2.waitKey(10) & 0xFF

    # break out of stream loop if esc or 'q' is pressed
	if key == 27 or key == ord('q'):        
		break
	
	# write output frame
	out_video.write(frame)

	# update fps counter
	fps.update()

	frame_nb += 1
	
# stop timer and display FPS information
fps.stop()
print("[INFO] elasped time: {:.2f}".format(fps.elapsed()))
print("[INFO] approx. FPS: {:.2f}".format(fps.fps()))

# close windows
cv2.destroyAllWindows()
cv2.waitKey(1) # workaround to effectively close window on mac

# release video stream & video rendering
fvs.stop()
out_video.release()

number of frames = 247
fps = 23.976
video width = 1280
video height = 720
[INFO] starting video file thread...
Ignoring empty camera frame.
[INFO] elasped time: 4.82
[INFO] approx. FPS: 51.19
